# Published SAGE inference reference
No training, field-data evaluation, or competition submission. Full local comparison is required after collection.

In [ ]:
import base64, hashlib, io, json, signal, time, urllib.request
from pathlib import Path
import numpy as np
import torch
started = time.monotonic()
signal.signal(signal.SIGALRM, lambda *_: (_ for _ in ()).throw(TimeoutError('20-minute reference budget exceeded')))
signal.alarm(1200)
assert torch.cuda.is_available(), 'Actual CUDA GPU required'
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
device = torch.device('cuda')
"""Minimal inference-only adapters matching the published SAGE networks."""
import ast
from pathlib import Path
import zipfile

import numpy as np
import torch
from torch import nn


class MLPBlock(nn.Module):
    def __init__(self,width):
        super().__init__();self.layers=nn.Sequential(nn.Linear(width,width),nn.ReLU())
    def forward(self,x):return self.layers(x)


class SimpleMLP(nn.Module):
    def __init__(self,num_classes,hidden_dim=256,num_blocks=4,use_location=False,input_dim=52):
        super().__init__()
        if use_location:raise ValueError('Only the admitted environment-only network is implemented')
        self.net=nn.Sequential(nn.Linear(input_dim,hidden_dim),nn.ReLU(),*[MLPBlock(hidden_dim) for _ in range(num_blocks)])
        self.classifier=nn.Linear(hidden_dim,num_classes)
    def forward(self,x):return self.classifier(self.net(x))


class ResidualBlock(nn.Module):
    def __init__(self,width,inner,dropout1,dropout2):
        super().__init__();self.norm=nn.BatchNorm1d(width);self.fc1=nn.Linear(width,inner)
        self.act=nn.ReLU();self.drop1=nn.Dropout(dropout1);self.fc2=nn.Linear(inner,width);self.drop2=nn.Dropout(dropout2)
    def forward(self,x):return x+self.drop2(self.fc2(self.drop1(self.act(self.fc1(self.norm(x))))))


class ResNet(nn.Module):
    def __init__(self,num_classes,hidden_dim=256,num_blocks=4,d_hidden_multiplier=2.,
                 use_location=False,dropout1=0.,dropout2=0.,input_dim=52):
        super().__init__()
        if use_location:raise ValueError('Only the admitted environment-only network is implemented')
        self.projection=nn.Linear(input_dim,hidden_dim)
        self.blocks=nn.Sequential(*[ResidualBlock(hidden_dim,int(hidden_dim*d_hidden_multiplier),dropout1,dropout2) for _ in range(num_blocks)])
        self.head_norm=nn.BatchNorm1d(hidden_dim);self.head_act=nn.ReLU();self.classifier=nn.Linear(hidden_dim,num_classes)
    def forward(self,x):return self.classifier(self.head_act(self.head_norm(self.blocks(self.projection(x)))))


def normalization_from_source(path,predictors_subset):
    """Read literal constants from publisher code without importing/executing it."""
    tree=ast.parse(Path(path).read_text());statistics={}
    for node in tree.body:
        if not isinstance(node,ast.Assign) or not isinstance(node.value,ast.Dict):continue
        names=[target.id for target in node.targets if isinstance(target,ast.Name)]
        for name in names:
            if name not in ['predictors_means','predictors_stds']:continue
            values={}
            for key,value in zip(node.value.keys,node.value.values):
                group=ast.literal_eval(key)
                if group in predictors_subset and group!='location':values[group]=ast.literal_eval(value)
            statistics[name]=values
    arrays=[]
    for name in ['predictors_means','predictors_stds']:
        flat=[statistics[name][group][i] for group,indices in predictors_subset.items() if group!='location' for i in indices]
        arrays.append(np.array(flat,dtype=np.float32))
    if len(arrays[0])!=52 or np.any(arrays[1]<=0):raise ValueError('Published normalization contract mismatch')
    return arrays


def normalize(features,means,stds):
    x=torch.as_tensor(features,dtype=torch.float32)
    mean=torch.as_tensor(means,dtype=torch.float32,device=x.device)
    std=torch.as_tensor(stds,dtype=torch.float32,device=x.device)
    if x.ndim!=2 or x.shape[1]!=len(mean):raise ValueError('Feature count mismatch')
    result=(torch.where(torch.isnan(x),mean,x)-mean)/std
    if not torch.isfinite(result).all():raise ValueError('Nonfinite predictor remains after published normalization')
    return result


def load_network(checkpoint,model_config):
    if not zipfile.is_zipfile(checkpoint):raise ValueError('ZIP-serialized weights-only checkpoint required')
    loaded=torch.load(checkpoint,map_location='cpu',weights_only=True)
    state=loaded.get('state_dict',loaded)
    weights={k.removeprefix('net.'):v for k,v in state.items() if k.startswith('net.')}
    net=model_config['model']['init_args']['net'];kind=net['class_path'].split('.')[-1]
    if kind not in ['SimpleMLP','ResNet']:raise ValueError('Unadmitted model class')
    model=(SimpleMLP if kind=='SimpleMLP' else ResNet)(**net['init_args'])
    model.load_state_dict(weights,strict=True);model.eval()
    return model

bundle = np.load(io.BytesIO(base64.b64decode('...')))
members = {'aggregation_s0': {'name': 'aggregation_s0/checkpoints/best.ckpt', 'size': 48479690, 'data_offset': 1024, 'header_sha256': '028fe32d800a39355aa2dc711a4066f6b80b3180ab29428f64c1ae13ce0be56c', 'sha256': '771055f5ab82d32f2923d1c2aeb78b0e534697b01a67a32c80a27279c7659454'}}
configs = json.loads('{"aggregation_s0": {"model": {"init_args": {"net": {"class_path": "models.SimpleMLP", "init_args": {"num_classes": 5771, "hidden_dim": 512, "num_blocks": 4, "use_location": false}}}}}}')
archive_url = 'https://zenodo.org/api/records/21335338/files/checkpoints.tar/content'
outputs = {'cell_ids': bundle['cell_ids']}
reports = []
for run, member in members.items():
    path = Path('/kaggle/working') / (run + '.ckpt')
    digest = hashlib.sha256()
    with path.open('wb') as target:
        for offset in range(0, member['size'], 8 * 1024**2):
            count = min(8 * 1024**2, member['size'] - offset)
            left = member['data_offset'] + offset
            request = urllib.request.Request(archive_url, headers={'Range': f'bytes={left}-{left+count-1}'})
            with urllib.request.urlopen(request, timeout=40) as response:
                assert response.status == 206
                assert response.headers.get('Content-Range', '').startswith(f'bytes {left}-{left+count-1}/')
                body = response.read(count + 1)
            assert len(body) == count
            target.write(body); digest.update(body)
    assert digest.hexdigest() == member['sha256'], 'Checkpoint digest mismatch'
    model = load_network(path, configs[run]).to(device).eval()
    x = normalize(bundle[run+'__raw'], bundle[run+'__means'], bundle[run+'__stds']).to(device)
    torch.cuda.synchronize(); t0 = time.monotonic()
    with torch.inference_mode():
        prediction = torch.sigmoid(model(x)).cpu().numpy()
    torch.cuda.synchronize()
    elapsed = time.monotonic() - t0
    delta = np.abs(prediction[:, bundle['probe_columns']].astype(np.float64) - bundle[run+'__probe'])
    passed = bool(np.isfinite(prediction).all() and delta.max() <= 1e-5 and delta.mean() <= 1e-6)
    outputs[run] = prediction
    reports.append({'run': run, 'shape': list(prediction.shape), 'checkpoint_sha256': digest.hexdigest(),
                    'probe_max_absolute_error': float(delta.max()), 'probe_mean_absolute_error': float(delta.mean()),
                    'probe_passed': passed, 'inference_seconds': elapsed})
    path.unlink()
    del model
    torch.cuda.empty_cache()
np.savez_compressed('/kaggle/working/gpu_probabilities.npz', **outputs)
report = {'status': 'probe_passed_pending_full_local_comparison' if all(r['probe_passed'] for r in reports) else 'failed',
          'GPU_name': torch.cuda.get_device_name(), 'torch_version': torch.__version__,
          'CUDA_version': torch.version.cuda, 'elapsed_seconds': time.monotonic()-started,
          'runs': reports, 'neural_training_performed': False, 'target_labels_read': False,
          'NEON_credential_included': False, 'NEON_performance_evaluated': False}
Path('/kaggle/working/gpu_report.json').write_text(json.dumps(report, indent=2)+'\n')
print(json.dumps(report))
signal.alarm(0)
assert all(r['probe_passed'] for r in reports)
